In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [4]:
analize = Analizer(0.9)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_1_9_10,0.999982,0.632739,0.999979,0.999976,0.999977,0.000010,0.218022,3.769856e-06,0.000033,0.000018,0.000919,0.003229,1.000021,0.003366,110.942417,164.572954,"Hidden Size=[5, 4], regularizer=0.5, learning_..."
1,model_1_9_11,0.999981,0.632175,0.999947,0.999973,0.999971,0.000011,0.218357,9.519488e-06,0.000037,0.000023,0.000918,0.003339,1.000023,0.003481,110.808710,164.439246,"Hidden Size=[5, 4], regularizer=0.5, learning_..."
2,model_1_9_9,0.999981,0.633363,0.999996,0.999975,0.999978,0.000011,0.217651,7.550528e-07,0.000035,0.000018,0.000952,0.003358,1.000023,0.003501,110.785815,164.416351,"Hidden Size=[5, 4], regularizer=0.5, learning_..."
3,model_15_8_5,0.999978,0.714994,0.999586,0.999994,0.999903,0.000013,0.169192,7.682722e-05,0.000003,0.000040,0.001306,0.003583,1.000019,0.003735,124.526590,186.689257,"Hidden Size=[6, 4], regularizer=0.03, learning..."
4,model_15_8_1,0.999978,0.714994,0.999586,0.999994,0.999903,0.000013,0.169192,7.682807e-05,0.000003,0.000040,0.001306,0.003583,1.000019,0.003735,124.526553,186.689220,"Hidden Size=[6, 4], regularizer=0.03, learning..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4975,model_28_1_1,0.981672,0.585035,0.996115,0.985484,0.992398,0.010880,0.246341,1.166272e-03,0.005377,0.003272,0.046947,0.104308,1.012937,0.108748,125.041633,195.736431,"Hidden Size=[7, 4], regularizer=0.03, learning..."
4979,model_2_2_0,0.981426,0.712725,0.993422,0.990409,0.992378,0.011027,0.170539,5.372299e-03,0.004283,0.004828,0.050485,0.105007,1.022289,0.109478,97.014900,150.645436,"Hidden Size=[5, 4], regularizer=0.5, learning_..."
4983,model_23_5_2,0.981368,0.643027,0.991694,0.998460,0.996263,0.011061,0.211915,4.014100e-03,0.001115,0.002565,0.022996,0.105169,1.008599,0.109647,161.008733,253.643295,"Hidden Size=[7, 6], regularizer=0.03, learning..."
5006,model_23_4_0,0.980428,0.710457,0.998923,0.994288,0.995836,0.011619,0.171885,1.738893e-04,0.002783,0.001478,0.030855,0.107790,1.009033,0.112379,160.910287,253.544850,"Hidden Size=[7, 6], regularizer=0.03, learning..."
